### 00 - Train Models (Target + Shadow)

Trains the target (victim) model and N shadow (attacker proxy) models using a pure-contrastive Siamese encoder + MLP classifier head on ClinicalBERT embeddings.

##### Inputs
- `data/external/BaselineDataSplits/{target,shadow}_{train,test}.csv`
- `data/external/clinicalbert/{x1,x2,y}_{target,shadow}_{train,test}.npy`

##### Outputs
- `outputs/models/target_encoder.h5`, `target_clf.h5`
- `outputs/models/shadow_encoder_{i}.h5`, `shadow_clf_{i}.h5`
- `outputs/models/shadow_{x1,x2,y}_train_{i}.npy`
- `outputs/models/test_fold_indices.npy`
- `outputs/models/shadow_accuracy.csv`

In [ ]:
import warnings
from time import time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

from pprl_attack import (
    DATA_DIR, EMB_DIR, MODEL_DIR,
    N_SHADOWS, RANDOM_STATE,
    SA_EPOCHS, SA_BATCH_SIZE,
    CLF_EPOCHS, CLF_BATCH_SIZE, MARGIN, ENCODER_OUTPUT_DIM,
    build_encoder, contrastive_loss, build_classifier,
    set_seeds, ensure_dir, stratified_bootstrap_sample,
)

set_seeds(RANDOM_STATE)
ensure_dir(MODEL_DIR)

In [ ]:
# Load target data
target_train = pd.read_csv(DATA_DIR / "target_train.csv")
target_test = pd.read_csv(DATA_DIR / "target_test.csv")

x1_t_tr = np.load(EMB_DIR / "x1_target_train.npy")
x2_t_tr = np.load(EMB_DIR / "x2_target_train.npy")
y_t_tr  = np.load(EMB_DIR / "y_target_train.npy")
x1_t_te = np.load(EMB_DIR / "x1_target_test.npy")
x2_t_te = np.load(EMB_DIR / "x2_target_test.npy")
y_t_te  = np.load(EMB_DIR / "y_target_test.npy")

# Load shadow data
shadow_train = pd.read_csv(DATA_DIR / "shadow_train.csv")
shadow_test = pd.read_csv(DATA_DIR / "shadow_test.csv")

x1_s_tr_full = np.load(EMB_DIR / "x1_shadow_train.npy")
x2_s_tr_full = np.load(EMB_DIR / "x2_shadow_train.npy")
y_s_tr_full  = np.load(EMB_DIR / "y_shadow_train.npy")
x1_s_te = np.load(EMB_DIR / "x1_shadow_test.npy")
x2_s_te = np.load(EMB_DIR / "x2_shadow_test.npy")
y_s_te  = np.load(EMB_DIR / "y_shadow_test.npy")

print(f"Target train: {len(target_train)} pairs, labels: {target_train['label'].value_counts().to_dict()}")
print(f"Target test:  {len(target_test)} pairs, labels: {target_test['label'].value_counts().to_dict()}")
print(f"Shadow train: {len(shadow_train)} pairs, labels: {shadow_train['label'].value_counts().to_dict()}")
print(f"Shadow test:  {len(shadow_test)} pairs, labels: {shadow_test['label'].value_counts().to_dict()}")

In [ ]:
# ====================================
# Train Target Model
# ====================================

target_total = SA_EPOCHS + CLF_EPOCHS
target_done = [0]

class ProgressCallback(tf.keras.callbacks.Callback):
    def __init__(self, total, done):
        super().__init__()
        self.total = total
        self.done = done
    def on_epoch_end(self, epoch, logs=None):
        self.done[0] += 1
        pct = 100 * self.done[0] / self.total
        print(f"\r  Overall progress: {pct:.1f}%  ({self.done[0]}/{self.total} epochs)",
              end="", flush=True)

embedding_dim = x1_t_tr.shape[1]

sa_model, target_encoder = build_encoder(embedding_dim, output_dim=ENCODER_OUTPUT_DIM)
sa_model.compile(optimizer="adam", loss=contrastive_loss(margin=MARGIN))
sa_model.fit(
    [x1_t_tr, x2_t_tr], y_t_tr,
    epochs=SA_EPOCHS, batch_size=SA_BATCH_SIZE, validation_split=0.1,
    verbose=0, callbacks=[ProgressCallback(target_total, target_done)],
)

e1_tr = target_encoder.predict(x1_t_tr, verbose=0)
e2_tr = target_encoder.predict(x2_t_tr, verbose=0)
diff_tr = np.abs(e1_tr - e2_tr)

e1_te = target_encoder.predict(x1_t_te, verbose=0)
e2_te = target_encoder.predict(x2_t_te, verbose=0)
diff_te = np.abs(e1_te - e2_te)

target_clf = build_classifier(diff_tr.shape[1])
target_clf.fit(
    diff_tr, y_t_tr,
    epochs=CLF_EPOCHS, batch_size=CLF_BATCH_SIZE, validation_split=0.1,
    verbose=0, callbacks=[ProgressCallback(target_total, target_done)],
)
print()

y_prob_te = target_clf.predict(diff_te, verbose=0).flatten()
y_pred_te = (y_prob_te > 0.5).astype(int)
y_prob_tr = target_clf.predict(diff_tr, verbose=0).flatten()
y_pred_tr = (y_prob_tr > 0.5).astype(int)

targ_test_acc = accuracy_score(y_t_te, y_pred_te)
targ_train_acc = accuracy_score(y_t_tr, y_pred_tr)

print(f"Target model:")
print(f"  Train accuracy: {targ_train_acc:.4f}")
print(f"  Test accuracy:  {targ_test_acc:.4f}")
print(f"  Gap:            {targ_train_acc - targ_test_acc:.4f}")

target_encoder.save(MODEL_DIR / "target_encoder.h5")
target_clf.save(MODEL_DIR / "target_clf.h5")
np.save(MODEL_DIR / "target_train_probs.npy", y_prob_tr)
np.save(MODEL_DIR / "target_test_probs.npy", y_prob_te)
print("Target model saved.")

In [ ]:
# ====================================
# Prepare disjoint test folds for shadows
# ====================================
skf = StratifiedKFold(n_splits=N_SHADOWS, shuffle=True, random_state=RANDOM_STATE)
test_folds = list(skf.split(np.zeros(len(y_s_te)), y_s_te))
te_fold_indices = np.array([fold[1] for fold in test_folds])
np.save(MODEL_DIR / "test_fold_indices.npy", te_fold_indices)
print(f"Test set split into {N_SHADOWS} folds (~{len(test_folds[0][1])} each).")

# ====================================
# Train all shadow models
# ====================================
embedding_dim = x1_s_tr_full.shape[1]
boot_indices = stratified_bootstrap_sample(shadow_train, N_SHADOWS, RANDOM_STATE)

shadow_total = N_SHADOWS * (SA_EPOCHS + CLF_EPOCHS)
shadow_done = [0]

total_start = time()
summary = []

for i in range(N_SHADOWS):
    t0 = time()
    idx = boot_indices[i]

    x1_tr = x1_s_tr_full[idx]
    x2_tr = x2_s_tr_full[idx]
    y_tr  = y_s_tr_full[idx]
    np.save(MODEL_DIR / f"shadow_x1_train_{i}.npy", x1_tr)
    np.save(MODEL_DIR / f"shadow_x2_train_{i}.npy", x2_tr)
    np.save(MODEL_DIR / f"shadow_y_train_{i}.npy",  y_tr)

    sa_model, encoder = build_encoder(embedding_dim, output_dim=ENCODER_OUTPUT_DIM)
    sa_model.compile(optimizer="adam", loss=contrastive_loss(margin=MARGIN))
    sa_model.fit(
        [x1_tr, x2_tr], y_tr,
        epochs=SA_EPOCHS, batch_size=SA_BATCH_SIZE,
        validation_split=0.1,
        verbose=0, callbacks=[ProgressCallback(shadow_total, shadow_done)],
    )

    enc1_tr = encoder.predict(x1_tr, verbose=0)
    enc2_tr = encoder.predict(x2_tr, verbose=0)
    diff_tr = np.abs(enc1_tr - enc2_tr)

    clf = build_classifier(diff_tr.shape[1])
    clf.fit(
        diff_tr, y_tr,
        epochs=CLF_EPOCHS, batch_size=CLF_BATCH_SIZE,
        validation_split=0.1,
        verbose=0, callbacks=[ProgressCallback(shadow_total, shadow_done)],
    )

    y_pred_tr = (clf.predict(diff_tr, verbose=0).flatten() > 0.5).astype(int)
    acc_tr = accuracy_score(y_tr, y_pred_tr)

    te_idx = test_folds[i][1]
    enc1_te = encoder.predict(x1_s_te[te_idx], verbose=0)
    enc2_te = encoder.predict(x2_s_te[te_idx], verbose=0)
    diff_te = np.abs(enc1_te - enc2_te)
    y_pred_te = (clf.predict(diff_te, verbose=0).flatten() > 0.5).astype(int)
    acc_te = accuracy_score(y_s_te[te_idx], y_pred_te)

    elapsed = time() - t0
    summary.append({
        "shadow": i, "train_acc": acc_tr, "test_acc": acc_te, "time_s": int(elapsed),
    })
    print(f"\n  [{i+1}/{N_SHADOWS}] train={acc_tr:.4f}  test={acc_te:.4f}  ({elapsed:.0f}s)")

    encoder.save(MODEL_DIR / f"shadow_encoder_{i}.h5")
    clf.save(MODEL_DIR / f"shadow_clf_{i}.h5")

total_elapsed = time() - total_start
print(f"\nAll {N_SHADOWS} shadow models trained in {total_elapsed:.0f}s ({total_elapsed/60:.1f}min).")
print("\nSummary:")
summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(MODEL_DIR / "shadow_accuracy.csv", index=False)